In [2]:
import json
import re
import unicodedata

In [4]:
def normalize_and_clean(text):
    if not text:
        return ""
    
    # 1. Token Normalization: Fix unicode anomalies (e.g., converting "smart" quotes to straight quotes)
    # This is crucial for character-level models to avoid expanding the vocabulary unnecessarily.
    text = unicodedata.normalize('NFKC', text)
    '''NFKC (Unicode Normalization Form Compatibility Composition) is a text
    preprocessing technique that standardizes strings by replacing full-width
    characters and compatibility symbols with their standard versions, which
    helps clean text data before training an ML model.'''
    text = text.replace('‘', "'").replace('’', "'").replace('“', '"').replace('”', '"')
    text = text.replace('—', '-')
    
    # 2. Formatting Cleanup: Remove excessive spaces but preserve intended newlines
    text = re.sub(r'[ \t]+', ' ', text)
    
    return text.strip()

In [5]:
def process_dataset(name, raw_data, text_key = 'text'):
    print(f"--- Cleaning Corpus: {name} ---")
    seen_texts = set()
    clean_data = []
    
    duplicate_count = 0
    malformed_count = 0
    
    for entry in raw_data:
        raw_text = entry.get(text_key, '')
        
        # Apply cleaning and normalization
        cleaned_text = normalize_and_clean(raw_text)
        
        # 3. Removal of malformed entries: Skip if text is unusually short (e.g., < 40 chars)
        if len(cleaned_text) < 40:
            malformed_count += 1
            continue
        
        # 4. Deduplication: Check if we've already seen this exact text
        if cleaned_text in seen_texts:
            duplicate_count += 1
            continue
        seen_texts.add(cleaned_text)
        
        # Update entry and save
        entry[text_key] = cleaned_text
        clean_data.append(entry)
        
    print(f"Original size: {len(raw_data)}")
    print(f"Removed duplicates: {duplicate_count}")
    print(f"Removed malformed/short: {malformed_count}")
    print(f"Final clean size: {len(clean_data)}\n")
    
    return clean_data

In [7]:
# Execution

# 1. Load, Clean and Save Poems
poems_raw = []
with open('/home/naman/Cryptonite-JM-Tasks/GenAI/Task-4/Project/data/poems_pretraining_corpus.jsonl', 'r', encoding = 'utf-8') as f:
    for line in f:
        poems_raw.append(json.loads(line))

clean_poems = process_dataset("Pre-training Poems", poems_raw, text_key ='text')

with open('/home/naman/Cryptonite-JM-Tasks/GenAI/Task-4/Project/data/clean_poems_corpus.jsonl', 'w', encoding='utf-8') as f:
    for entry in clean_poems:
        f.write(json.dumps(entry) + '\n')

# 2. Load, Clean, and Save Raps
with open('/home/naman/Cryptonite-JM-Tasks/GenAI/Task-4/Project/data/tagged_rap_dataset_gemma4.json', 'r', encoding='utf-8') as f:
    raps_raw = json.load(f)

clean_raps = process_dataset("Fine-tuning Raps", raps_raw, text_key='lyrics')

with open('/home/naman/Cryptonite-JM-Tasks/GenAI/Task-4/Project/data/clean_rap_dataset.json', 'w', encoding='utf-8') as f:
    json.dump(clean_raps, f, indent=4)

--- Cleaning Corpus: Pre-training Poems ---
Original size: 20656
Removed duplicates: 985
Removed malformed/short: 0
Final clean size: 19671

--- Cleaning Corpus: Fine-tuning Raps ---
Original size: 500
Removed duplicates: 0
Removed malformed/short: 0
Final clean size: 500

